# Cleaning & Harmonization — World Happiness Report 2015–2019 (Part A, Step 2)

**Objective:** turn the 5 raw CSVs into ONE clean dataset with the unified schema, applying the
cleaning rules proposed and verified in the EDA notebook, and save it to
`data/processed/happiness_unified.csv`.

**Cleaning decisions applied here — each one justified (self-contained):**

* **Missing values — drop, do NOT impute.** Exactly ONE cell of the 782 raw rows is
  missing (0.13 %): year 2018, `United Arab Emirates`, `corruption` (written `'N/A'` in the
  raw file). `corruption` is a subjective perception index that must not be fabricated by
  imputation, and at streaming inference the consumer has to reject any event that lacks a
  feature — so the model must not depend on any fill-in rule. The 31 published zero cells
  (e.g. Somalia `gdp`, Greece `generosity`) are KEPT: they are real floor values of the
  report, not missing-value sentinels.
* **Country names — normalize with ONE explicit dictionary.** 6 name variants of the same
  country (12 cells across the 5 files, e.g. `Macedonia` vs `North Macedonia`) are merged
  so that `country` — the natural key and future Kafka key — stays stable. Every other
  name passes through unchanged; countries honestly not measured in all 5 years are kept
  and documented.
* **Columns dropped — target leakage + not part of the event contract.** The rank
  columns, `Standard Error`, the confidence intervals / whiskers and `Dystopia Residual`
  (literally the score minus the six factors) are removed here because they directly leak
  the target; `Region` exists only in 2015–2016 and is not part of the event contract
  either.

**Expected row accounting:** 782 raw rows → drop 1 → **781 final rows**, key = (`country`, `year`).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# Project root (this notebook lives in notebooks/ under the repo root)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

YEARS = [2015, 2016, 2017, 2018, 2019]
UNIFIED = ["country", "year", "gdp", "family", "health",
           "freedom", "generosity", "corruption", "happiness_score"]
# The 7 numeric columns (the target was never a model feature, but it stays in the
# dataset as the actual outcome for evaluation and for the fact table later)
NUMERIC = ["gdp", "family", "health", "freedom", "generosity", "corruption", "happiness_score"]

# Column names unified (selected BY NAME, never by position — column order differs per year)
MAP = {
    2015: {"Country": "country", "Happiness Score": "happiness_score",
           "Economy (GDP per Capita)": "gdp", "Family": "family",
           "Health (Life Expectancy)": "health", "Freedom": "freedom",
           "Trust (Government Corruption)": "corruption", "Generosity": "generosity"},
    2016: {"Country": "country", "Happiness Score": "happiness_score",
           "Economy (GDP per Capita)": "gdp", "Family": "family",
           "Health (Life Expectancy)": "health", "Freedom": "freedom",
           "Trust (Government Corruption)": "corruption", "Generosity": "generosity"},
    2017: {"Country": "country", "Happiness.Score": "happiness_score",
           "Economy..GDP.per.Capita.": "gdp", "Family": "family",
           "Health..Life.Expectancy.": "health", "Freedom": "freedom",
           "Trust..Government.Corruption.": "corruption", "Generosity": "generosity"},
    2018: {"Country or region": "country", "Score": "happiness_score",
           "GDP per capita": "gdp", "Social support": "family",
           "Healthy life expectancy": "health", "Freedom to make life choices": "freedom",
           "Perceptions of corruption": "corruption", "Generosity": "generosity"},
    2019: {"Country or region": "country", "Score": "happiness_score",
           "GDP per capita": "gdp", "Social support": "family",
           "Healthy life expectancy": "health", "Freedom to make life choices": "freedom",
           "Perceptions of corruption": "corruption", "Generosity": "generosity"},
}

# confirmed: country-name normalization — one explicit dictionary
RENAME_TO = {
    "Hong Kong S.A.R., China": "Hong Kong",      # 2017 variant of Hong Kong
    "Macedonia": "North Macedonia",              # official 2019 renaming
    "North Cyprus": "Northern Cyprus",           # spelling variant
    "Somaliland region": "Somaliland Region",    # case-only difference
    "Taiwan Province of China": "Taiwan",        # 2017 variant of Taiwan
    "Trinidad & Tobago": "Trinidad and Tobago",  # '&' vs 'and'
}

# confirmed: drop, do not impute (justified in the markdown above)
EXPECTED_DROP = {"country": "United Arab Emirates", "year": 2018, "column": "corruption"}
EXPECTED_ROWS = {2015: 158, 2016: 157, 2017: 155, 2018: 156, 2019: 156}

print("ROOT =", ROOT)
print("Raw files present:", sorted(p.name for p in RAW.glob("*.csv")))

ROOT = C:\Users\juans\Downloads\ETL - Trabajos\Corte3\Workshop-3
Raw files present: ['2015.csv', '2016.csv', '2017.csv', '2018.csv', '2019.csv']


## 1. Reusable cleaning function

All the cleaning logic lives in ONE function so the step is reproducible and the
downstream tasks can reuse the exact same rules. It returns the unified DataFrame and a
row-by-step accounting log.

**Cleaning order (each step logs its row count):**

1. Read each file by name; add `year` from the file name.
2. Rename columns to the unified names and keep only the 9 unified columns.
   The dropped rank / `Standard Error` / confidence-intervals / whiskers /
   `Dystopia Residual` columns leak the target, and `Region` exists only in 2015–2016 —
   justification in the markdown above.
3. Cast types: `country` → stripped string, `year` → int64, features & target → float64.
4. Normalize country names with the explicit `RENAME_TO` dictionary only.
5. Concatenate the 5 years.
6. Drop the single row with a missing feature value and log it (why: the
   missing-value justification above).

Nothing else is altered: published zeros are kept, countries measured in fewer than the
5 years are kept, no outliers are removed.

In [2]:
def build_happiness_unified(raw_dir: Path):
    """Load the 5 raw CSVs and apply the confirmed cleaning rules.

    Returns (unified_df, accounting_df, renamed_cells, dropped_row_dict). The accounting
    DataFrame reports the row count after every step, so the final total is explainable.
    """
    log = []

    # --- 1) load each file BY NAME and validate ITS OWN schema before touching anything ---
    frames = {}          # year -> raw dataframe of that single CSV
    for year in YEARS:
        df = pd.read_csv(raw_dir / f"{year}.csv")
        assert len(df) == EXPECTED_ROWS[year], f"{year}: unexpected row count {len(df)}"
        assert "year" not in df.columns, f"{year}: raw file unexpectedly has a year column"
        # every mapping key must exist BY NAME in THIS file's own columns: each year has a
        # different schema, so checking against a 5-file column union would hide a
        # column that is missing in one specific year
        missing_maps = set(MAP[year]) - set(df.columns)
        assert not missing_maps, f"{year}: mapping keys absent: {sorted(missing_maps)}"
        frames[year] = df.assign(year=year)
    log.append({"step": "1. load 5 raw files (year from file name; per-file schema guard)",
                "rows": sum(len(df) for df in frames.values())})

    # --- 2) rename + select the 9 unified columns PER FILE, then concatenate ONCE ---------
    parts = [frames[year].rename(columns=MAP[year])[UNIFIED] for year in YEARS]
    unified = pd.concat(parts, ignore_index=True)
    log.append({"step": "2. rename + select 9 unified columns per file "
                        "(leakage columns dropped); concatenate once",
                "rows": len(unified)})

    # --- 3) cast types --------------------------------------------------------------------
    unified["country"] = unified["country"].astype(str).str.strip()
    unified["year"] = unified["year"].astype("int64")
    for col in NUMERIC:
        unified[col] = unified[col].astype("float64")
    log.append({"step": "3. cast types (country str / year int64 / 7 float64)",
                "rows": len(unified)})

    # --- 4) normalize country names ---------------------------------------------------------
    before = unified["country"]
    unified["country"] = unified["country"].map(lambda s: RENAME_TO.get(s, s))
    renamed_cells = int((before != unified["country"]).sum())
    log.append({"step": f"4. country normalization (RENAME_TO; {renamed_cells} cells changed)",
                "rows": len(unified)})

    # --- 5) uniqueness check on the natural key (concatenation happened in step 2) ---------
    dup_keys = int(unified.duplicated(subset=["country", "year"]).sum())
    assert dup_keys == 0, f"{dup_keys} duplicated (country, year) keys"
    log.append({"step": "5. (country, year) key unique (verified on the concatenated table)",
                "rows": len(unified)})

    # --- 6) drop the single row with a missing feature value (justified above) -------------
    feature_cols = [c for c in NUMERIC if c != "happiness_score"]
    missing_mask = unified[feature_cols].isna().any(axis=1)
    dropped_rows = unified[missing_mask]
    assert len(dropped_rows) == 1, f"expected exactly 1 row with a missing feature, got {len(dropped_rows)}"
    drop_row = dropped_rows.iloc[0]
    assert (drop_row["country"] == EXPECTED_DROP["country"]
            and drop_row["year"] == EXPECTED_DROP["year"]
            and bool(pd.isna(drop_row[EXPECTED_DROP["column"]]))), (
        f"unexpected missing row: {drop_row.to_dict()}")
    unified = unified[~missing_mask].reset_index(drop=True)
    log.append({"step": f"6. drop the 1 row with a missing feature "
                        f"({drop_row['country']} {drop_row['year']}, "
                        f"missing '{EXPECTED_DROP['column']}')",
                "rows": len(unified)})

    accounting = pd.DataFrame(log)
    return unified, accounting, renamed_cells, drop_row.to_dict()

### 1.1 Run the cleaning and show the row accounting

782 raw rows must end up as **781** clean rows.

In [3]:
unified, accounting, renamed_cells, dropped = build_happiness_unified(RAW)
accounting.index += 1
accounting.index.name = "order"
print("dropped row (missing-value rule):", {k: dropped[k] for k in ("country", "year")})
print("missing column in dropped row:", EXPECTED_DROP["column"])
display(accounting)

dropped row (missing-value rule): {'country': 'United Arab Emirates', 'year': 2018}
missing column in dropped row: corruption


,step,rows
order,,
1,1. load 5 raw files (year from file name; per-...,782
2,2. rename + select 9 unified columns per file ...,782
3,3. cast types (country str / year int64 / 7 fl...,782
4,4. country normalization (RENAME_TO; 12 cells ...,782
5,"5. (country, year) key unique (verified on the...",782
6,6. drop the 1 row with a missing feature (Unit...,781


## 2. Final schema validation

Checks on the in-memory result before writing any file: exact column order, exact dtypes
(no column may fall back to `object`), unique key `('country', 'year')`, no missing values,
no negative values, 164 canonical countries, and the published zeros still present.

In [4]:
print("shape:", unified.shape)
print()
print("dtypes:")
print(unified.dtypes.to_string())
print()
print("head:")
display(unified.head())
print("tail (2019):")
display(unified.tail())

shape: (781, 9)

dtypes:
country             object
year                 int64
gdp                float64
family             float64
health             float64
freedom            float64
generosity         float64
corruption         float64
happiness_score    float64

head:


,country,year,gdp,family,health,freedom,generosity,corruption,happiness_score
0,Switzerland,2015,1.39651,1.34951,0.94143,0.66557,0.29678,0.41978,7.587
1,Iceland,2015,1.30232,1.40223,0.94784,0.62877,0.43630,0.14145,7.561
2,Denmark,2015,1.32548,1.36058,0.87464,0.64938,0.34139,0.48357,7.527
3,Norway,2015,1.45900,1.33095,0.88521,0.66973,0.34699,0.36503,7.522
4,Canada,2015,1.32629,1.32261,0.90563,0.63297,0.45811,0.32957,7.427


tail (2019):


,country,year,gdp,family,health,freedom,generosity,corruption,happiness_score
776,Rwanda,2019,0.359,0.711,0.614,0.555,0.217,0.411,3.334
777,Tanzania,2019,0.476,0.885,0.499,0.417,0.276,0.147,3.231
778,Afghanistan,2019,0.350,0.517,0.361,0.000,0.158,0.025,3.203
779,Central African Republic,2019,0.026,0.000,0.105,0.225,0.235,0.035,3.083
780,South Sudan,2019,0.306,0.575,0.295,0.010,0.202,0.091,2.853


In [5]:
# missing values after cleaning: the drop rule leaves NOTHING behind
missing_after = unified.isna().sum()
negative = (unified[NUMERIC] < 0).sum().sum()
zero_cells = int((unified[NUMERIC] == 0).sum().sum())

print("missing values per column after cleaning:", int(missing_after.sum()),
      "(expected 0)")
print("negative values:", int(negative), "(expected 0)")
print("published zero cells kept (real floor values, not missing-value sentinels):", zero_cells)

# spot check of the two documented zero families (see EDA §4.2)
greece = unified[unified["country"] == "Greece"][["year", "generosity"]]
somalia = unified[unified["country"] == "Somalia"][["year", "gdp"]]
display(somalia[somalia["year"].isin([2016, 2017, 2018, 2019])])
print("Greece 'generosity' values:", sorted(greece["generosity"].unique().tolist()))

missing values per column after cleaning: 0 (expected 0)
negative values: 0 (expected 0)
published zero cells kept (real floor values, not missing-value sentinels): 31


,year,gdp
233,2016,0.000000
407,2017,0.022643
566,2018,0.000000
736,2019,0.000000


Greece 'generosity' values: [0.0]


In [6]:
# 164 canonical names after normalization. Countries with fewer than 5 ROWS after
# cleaning = 19: the 18 countries the raw surveys never measured in all 5 years (kept and
# documented, e.g. Somalia has no 2015 row) + United Arab Emirates, which WAS measured in
# all 5 years but lost its 2018 row when the missing 'corruption' cell was dropped.
by_year = unified.groupby("year")["country"].nunique()
print("distinct countries per year:")
print(by_year.to_string())
print()
print("total distinct countries:", unified["country"].nunique(), "(expected 164)")
partial = unified.groupby("country")["year"].nunique()
print("countries with < 5 rows after cleaning:", int((partial < 5).sum()),
      "= 18 absent from the raw surveys + United Arab Emirates (its 2018 row was the dropped missing-value row)")

distinct countries per year:
year
2015    158
2016    157
2017    155
2018    155
2019    156

total distinct countries: 164 (expected 164)
countries with < 5 rows after cleaning: 19 = 18 absent from the raw surveys + United Arab Emirates (its 2018 row was the dropped missing-value row)


## 3. Save the unified dataset → `data/processed/happiness_unified.csv`

Only this step writes into `data/processed/`. The file is the unique deliverable of the
cleaning step: `raw/` stays untouched (read-only), and later steps (train/test split,
streaming) must load THIS file, never the raw CSVs.

In [7]:
OUT_FILE = PROCESSED / "happiness_unified.csv"
unified.to_csv(OUT_FILE, index=False, encoding="utf-8", lineterminator="\n")
print(f"written: {OUT_FILE}")
print(f"size: {OUT_FILE.stat().st_size:,} bytes")

written: C:\Users\juans\Downloads\ETL - Trabajos\Corte3\Workshop-3\data\processed\happiness_unified.csv
size: 59,763 bytes


In [8]:
# round-trip sanity: the saved file must reload into the exact same clean table
reloaded = pd.read_csv(OUT_FILE)
print("reloaded shape:", reloaded.shape)
print(reloaded.dtypes.to_string())

reloaded shape: (781, 9)
country             object
year                 int64
gdp                float64
family             float64
health             float64
freedom            float64
generosity         float64
corruption         float64
happiness_score    float64


## 4. Final programmatic checks

In [9]:
checks = [
    ("782 raw rows loaded (158/157/155/156/156)", int(accounting.iloc[0]["rows"]) == 782),
    ("final columns exactly the 9 unified names in order", list(unified.columns) == UNIFIED),
    ("dtypes exact (country str, year int64, 7 float64)",
     all(isinstance(v, str) for v in unified["country"].head(20))
     and unified["country"].map(type).eq(str).all()
     and unified["year"].dtype == np.int64
     and all(unified[c].dtype == np.float64 for c in NUMERIC)),
    ("12 country cells normalized by RENAME_TO", renamed_cells == 12),
    ("no duplicated (country, year) keys", int(unified.duplicated(["country", "year"]).sum()) == 0),
    ("exactly 1 row dropped by the missing-value rule (2018 UAE corruption NaN)",
     dropped["country"] == "United Arab Emirates" and dropped["year"] == 2018
     and int(accounting.loc[accounting.index[-1], "rows"]) == 781),
    ("781 final rows", len(unified) == 781),
    ("no missing values after cleaning", missing_after.sum() == 0),
    ("no negative values", negative == 0),
    ("published zeros kept", zero_cells > 0
     and 0.0 in somalia["gdp"].values and 0.0 in greece["generosity"].values),
    ("164 canonical countries", unified["country"].nunique() == 164),
    ("CSV round-trip: same shape", reloaded.shape == (781, 9)),
    ("CSV round-trip: no missing values", int(reloaded.isna().sum().sum()) == 0),
]
final_checks = pd.DataFrame(checks, columns=["check", "result"])
final_checks["result"] = final_checks["result"].map({True: "PASS", False: "FAIL"})
print(final_checks.to_string(index=False))
assert final_checks["result"].eq("PASS").all(), "cleaning checks failed!"
print()
print("DONE: data/processed/happiness_unified.csv is ready for the next step.")

                                                                    check result
                                782 raw rows loaded (158/157/155/156/156)   PASS
                       final columns exactly the 9 unified names in order   PASS
                        dtypes exact (country str, year int64, 7 float64)   PASS
                                 12 country cells normalized by RENAME_TO   PASS
                                       no duplicated (country, year) keys   PASS
exactly 1 row dropped by the missing-value rule (2018 UAE corruption NaN)   PASS
                                                           781 final rows   PASS
                                         no missing values after cleaning   PASS
                                                       no negative values   PASS
                                                     published zeros kept   PASS
                                                  164 canonical countries   PASS
                            